# BdSLW60 - Week 1 pipeline (Kaggle)

**Dataset:** https://www.kaggle.com/datasets/hasaniut/bdslw60 (add it via *Add Input* -> search `hasaniut/bdslw60`)
**Authors' code (reference only):** https://github.com/hasanssl/BdSLW60_Code

**Notebook settings:** GPU T4 (needed only for the training cells), Internet **On** (to `pip install mediapipe` and download the Holistic model file).

**What this notebook does (Week 1 of the plan)**
1. Finds the videos and annotation files in the dataset and checks them against the paper (Table 6 trial counts).
2. Extracts MediaPipe Holistic keypoints (7 body + 2x21 hand nodes = 49) with a detection mask.
3. Builds the cached arrays: gap filling, shoulder normalisation, left-hand mirroring, resampling to T=64.
4. Writes the P1 split (train 15 signers / validate U5 / test U4+U8) and 18 leave-one-signer-out folds.
5. Trains a compact ST-GCN-style baseline for B0 (zero-filled) and B1 (+ gap filling), **validation signer only**.

**Read this first**
- The Kaggle page and the authors' repo README say the dataset ships **videos + annotation files (txt and JSON)**; the README describes landmark extraction as a separate step that took them 10+ hours. So this notebook extracts keypoints itself rather than loading ready-made arrays.
- I could not open the dataset's file listing or the JSON files, so I do not know the exact annotation schema. Cells 5-6 print the structure and guess the keys. If the report in cell 6 looks wrong (no segments, or counts far from 10-15 per video), copy the outputs of cells 3-6 and send them to me and I will fix the adapter.
- The model in the last cells is my own compact ST-GCN-style network. It is **not** the official SL-GCN from the SAM-SLR repo; swap that in next.
- If `pip install mediapipe` changes numpy/protobuf, restart the kernel once (Run -> Restart) and re-run from cell 2.
- The test signers U4/U8 are never evaluated in this notebook. Keep it that way until week 3.

In [1]:
# mediapipe >= 0.10.31 removed the legacy mp.solutions.holistic API (and 0.10.21 has no wheel for Python 3.13),
# so this notebook uses the Tasks HolisticLandmarker, which needs a model file.
!pip install -q mediapipe
import os, sys, urllib.request
print("python", sys.version.split()[0])
MODEL_PATH = '/kaggle/working/holistic_landmarker.task'
MODEL_URL = ('https://storage.googleapis.com/mediapipe-models/holistic_landmarker/'
             'holistic_landmarker/float16/latest/holistic_landmarker.task')
if not os.path.exists(MODEL_PATH):
    try:
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    except Exception as e:
        print('MODEL DOWNLOAD FAILED:', e, '\nDownload holistic_landmarker.task by hand, add it as a Kaggle dataset '
              'and set MODEL_PATH to its /kaggle/input/... path.')
if os.path.exists(MODEL_PATH):
    print('model file: %s (%.1f MB)' % (MODEL_PATH, os.path.getsize(MODEL_PATH) / 1e6))


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
python 3.13.15
model file: /kaggle/working/holistic_landmarker.task (13.7 MB)


In [2]:
import os, re, json, time, math, random, collections, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

WORK = Path('/kaggle/working')
CACHE = WORK / 'bdslw60_cache'
KP_DIR = CACHE / 'keypoints'
for d in (CACHE, KP_DIR):
    d.mkdir(parents=True, exist_ok=True)

T_OUT = 64              # frames after resampling
MAX_GAP = 5             # longest gap (frames) that is interpolated
N_NODES = 49            # 7 body + 21 left hand + 21 right hand
QUICK_TEST_VIDEOS = None  # e.g. 12 -> extract only 12 videos first to test everything quickly

# Word ids in class order (Table 4 of the paper): class index 0..59
WORD_IDS = (list(range(1, 13)) + [19, 20] + list(range(37, 51)) + list(range(91, 101))
            + [111, 112] + list(range(211, 221)) + list(range(351, 361)))
assert len(WORD_IDS) == 60
WORD2CLS = {w: i for i, w in enumerate(WORD_IDS)}

# Trials per signer from Table 6 of the paper (train + test columns), used as a sanity check
TABLE6 = {1: 975, 2: 744, 3: 583, 4: 637, 5: 600, 6: 815, 7: 107, 8: 639, 9: 656,
          10: 383, 11: 771, 12: 655, 13: 621, 14: 112, 15: 655, 16: 97, 17: 116, 18: 141}
assert sum(TABLE6.values()) == 9307

VID_RE = re.compile(r'U(\d+)W(\d+)([A-Za-z]*)', re.I)   # e.g. U11W1F.mp4
print('config ok')

config ok


In [3]:
# ---- Cell 3: find everything the dataset contains ----
INPUT_ROOT = Path('/kaggle/input')
assert INPUT_ROOT.exists() and any(INPUT_ROOT.iterdir()), 'No input found: use Add Input -> hasaniut/bdslw60'
all_files = [p for p in INPUT_ROOT.rglob('*') if p.is_file()]
print('files found:', len(all_files))
print('top-level inputs:', [p.name for p in INPUT_ROOT.iterdir()])
ext_count = collections.Counter(p.suffix.lower() for p in all_files)
print('extensions:', dict(ext_count.most_common(12)))
for ext in list(ext_count)[:8]:
    examples = [str(p) for p in all_files if p.suffix.lower() == ext][:3]
    print('\n' + (ext or '(no extension)') + ':')
    for e in examples:
        print('   ', e)

files found: 838
top-level inputs: ['datasets']
extensions: {'.mp4': 777, '.txt': 30, '.json': 30, '.pdf': 1}

.pdf:
    /kaggle/input/datasets/hasaniut/bdslw60/BdSLW60.pdf

.mp4:
    /kaggle/input/datasets/hasaniut/bdslw60/W1-2/U10W1F.mp4
    /kaggle/input/datasets/hasaniut/bdslw60/W1-2/U10W2F.mp4
    /kaggle/input/datasets/hasaniut/bdslw60/W1-2/U11W1F.mp4

.txt:
    /kaggle/input/datasets/hasaniut/bdslw60/W1-2/annotationW1-2.txt
    /kaggle/input/datasets/hasaniut/bdslw60/W11-12/annotationW11-12.txt
    /kaggle/input/datasets/hasaniut/bdslw60/W111-112/annotationW111-112.txt

.json:
    /kaggle/input/datasets/hasaniut/bdslw60/W1-2/output1.json
    /kaggle/input/datasets/hasaniut/bdslw60/W11-12/output1.json
    /kaggle/input/datasets/hasaniut/bdslw60/W111-112/output1.json


In [4]:
# ---- Cell 4: videos (user, word, view come from the file name, e.g. U11W1F.mp4) ----
rows = []
for p in all_files:
    if p.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv'):
        m = VID_RE.search(p.stem)
        if m:
            rows.append(dict(path=str(p), stem=p.stem, user=int(m[1]), word=int(m[2]), view=m[3].upper()))
videos = pd.DataFrame(rows)
assert len(videos) > 0, 'No videos matched the U<user>W<word><view> naming - send me the output of cell 3.'
print('video files by view:', videos.view.value_counts().to_dict())
if (videos.view == 'F').any():
    videos = videos[videos.view == 'F']
dups = videos.duplicated(['user', 'word']).sum()
if dups:
    print('WARNING:', dups, 'duplicate (user, word) videos - keeping the first of each')
    videos = videos.drop_duplicates(['user', 'word'])
videos = videos.reset_index(drop=True)
print('front-view videos:', len(videos), '| signers:', videos.user.nunique(), '| words:', videos.word.nunique())
unknown_words = sorted(set(videos.word) - set(WORD_IDS))
print('words not in the 60-word list:', unknown_words[:20], '(these will be ignored)' if unknown_words else '')
videos = videos[videos.word.isin(WORD_IDS)].reset_index(drop=True)
print('videos kept:', len(videos), '(60 words x 18 signers = 1080 would be complete)')

video files by view: {'F': 776, 'FL': 1}
front-view videos: 774 | signers: 18 | words: 60
words not in the 60-word list: [] 
videos kept: 774 (60 words x 18 signers = 1080 would be complete)


In [5]:
# ---- Cell 5: look at the annotation files ----
json_files = sorted([p for p in all_files if p.suffix.lower() == '.json'])
txt_files = sorted([p for p in all_files if p.suffix.lower() == '.txt'])
print(len(json_files), 'json files;', len(txt_files), 'txt files')
for p in json_files[:5]:
    print('  ', p)

def describe(obj, depth=0, max_depth=5, max_items=4):
    pad = '  ' * depth
    if depth > max_depth:
        print(pad + '...')
        return
    if isinstance(obj, dict):
        print(pad + 'dict(%d keys)' % len(obj))
        for k in list(obj)[:max_items]:
            print(pad + '  key %r:' % (k,))
            describe(obj[k], depth + 2, max_depth, max_items)
        if len(obj) > max_items:
            print(pad + '  ... +%d more keys' % (len(obj) - max_items))
    elif isinstance(obj, list):
        print(pad + 'list(len=%d)' % len(obj))
        for v in obj[:2]:
            describe(v, depth + 1, max_depth, max_items)
    else:
        print(pad + repr(obj)[:80])

if json_files:
    with open(json_files[0], encoding='utf-8') as f:
        sample_json = json.load(f)
    print('\nSTRUCTURE OF', json_files[0].name)
    describe(sample_json)
if txt_files:
    print('\nFIRST LINES OF', txt_files[0].name)
    with open(txt_files[0], errors='ignore') as f:
        print(''.join(f.readlines()[:15]))

30 json files; 30 txt files
   /kaggle/input/datasets/hasaniut/bdslw60/W1-2/output1.json
   /kaggle/input/datasets/hasaniut/bdslw60/W11-12/output1.json
   /kaggle/input/datasets/hasaniut/bdslw60/W111-112/output1.json
   /kaggle/input/datasets/hasaniut/bdslw60/W19-20/output1.json
   /kaggle/input/datasets/hasaniut/bdslw60/W211-212/output1.json

STRUCTURE OF output1.json
dict(2 keys)
  key 'W1':
    dict(13 keys)
      key 'U1':
        dict(1 keys)
          key 'LeftHand':
            ...
      key 'U2':
        dict(1 keys)
          key 'RightHand':
            ...
      key 'U3':
        dict(1 keys)
          key 'RightHand':
            ...
      key 'U4':
        dict(1 keys)
          key 'LeftHand':
            ...
      ... +9 more keys
  key 'W2':
    dict(13 keys)
      key 'U1':
        dict(1 keys)
          key 'LeftHand':
            ...
      key 'U2':
        dict(1 keys)
          key 'RightHand':
            ...
      key 'U3':
        dict(1 keys)
          key 'Rig

In [6]:
# ---- Cell 6: turn the annotations into one table of trials (video, start frame, end frame, hand) ----
START_KEY = None   # set these three if the auto-guess printed below is wrong, e.g. 'actual_start'
END_KEY = None     # e.g. 'end'
HAND_KEY = None    # e.g. 'hand'
HAND_MAP = {}      # extra raw-value -> 'L'/'R' mapping if the hand values are not words, e.g. {0: 'R', 1: 'L'}
ALLOW_UNKNOWN_HAND = False

def load_json(p):
    with open(p, encoding='utf-8') as f:
        return json.load(f)

def iter_dicts(obj, path=()):
    if isinstance(obj, dict):
        yield path, obj
        for k, v in obj.items():
            yield from iter_dicts(v, path + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_dicts(v, path + ('[%d]' % i,))

key_counter = collections.Counter()
for p in json_files[:3]:
    for _, d in iter_dicts(load_json(p)):
        key_counter.update(str(k) for k in d.keys())
print('most common keys in the first JSON files:', key_counter.most_common(25))

def pick_key(cands, prefer=(), avoid=()):
    cands = [k for k in cands if not any(a in k.lower() for a in avoid)]
    for pref in prefer:
        for k in cands:
            if pref in k.lower():
                return k
    return cands[0] if cands else None

keys = list(key_counter)
S_KEY = START_KEY or pick_key([k for k in keys if 'start' in k.lower()], prefer=('actual',))
E_KEY = END_KEY or pick_key([k for k in keys if 'end' in k.lower()], prefer=('end_frame', 'end'), avoid=('withdraw', 'extend'))
H_KEY = HAND_KEY or pick_key([k for k in keys if any(s in k.lower() for s in ('dominan', 'hand', 'orient'))], prefer=('dominan', 'hand'))
if H_KEY in ('LeftHand', 'RightHand'):   # these are tree levels (W1 -> U1 -> LeftHand), not fields
    H_KEY = None
print('using keys -> start:', S_KEY, '| end:', E_KEY, '| hand:', H_KEY)

def norm_hand(v):
    if v in HAND_MAP:
        return HAND_MAP[v]
    s = str(v).strip().lower()
    if s.startswith('l'):
        return 'L'
    if s.startswith('r'):
        return 'R'
    return None

def as_int(v):
    try:
        return int(float(v))
    except Exception:
        return None

def signature(path, d):
    parts = list(path) + [str(v) for v in d.values() if isinstance(v, (str, int))]
    return ' '.join(parts)

segs = []
for jp in json_files:
    for path, d in iter_dicts(load_json(jp)):
        if S_KEY not in d or E_KEY not in d:
            continue
        m = VID_RE.search(signature(path, d))
        view = ''
        if m:
            user, word, view = int(m[1]), int(m[2]), m[3].upper()
        else:
            uk = [k for k in d if 'user' in str(k).lower()]
            wk = [k for k in d if 'word' in str(k).lower()]
            user = as_int(re.sub(r'\D', '', str(d[uk[0]]))) if uk else None
            word = as_int(re.sub(r'\D', '', str(d[wk[0]]))) if wk else None
        if user is None or word is None or view not in ('', 'F'):
            continue
        starts, ends = d[S_KEY], d[E_KEY]
        path_hand = next((p for p in path if p in ('LeftHand', 'RightHand')), None)   # per-video hand from the JSON path
        hands = d.get(H_KEY) if H_KEY else path_hand
        if not isinstance(starts, list):
            starts, ends = [starts], [ends]
        if not isinstance(hands, list):
            hands = [hands] * len(starts)
        for i, (s, e) in enumerate(zip(starts, ends)):
            s, e = as_int(s), as_int(e)
            if s is None or e is None or e <= s:
                continue
            segs.append(dict(user=user, word=word, start=s, end=e,
                             hand_raw=hands[i] if i < len(hands) else None, src=jp.name))

segs = pd.DataFrame(segs)
print(len(segs), 'segments parsed')
assert len(segs) > 0, 'No segments parsed - send me the outputs of cells 5 and 6 so the adapter can be fixed.'
segs['hand'] = segs.hand_raw.map(norm_hand)
print('raw hand values:', segs.hand_raw.value_counts(dropna=False).head(8).to_dict())
n_unknown = int(segs.hand.isna().sum())
print('segments with unknown hand:', n_unknown)
if n_unknown and not ALLOW_UNKNOWN_HAND:
    raise RuntimeError('Hand dominance not found for some segments. Set HAND_KEY / HAND_MAP above, '
                       'or set ALLOW_UNKNOWN_HAND = True to treat them as right-handed (not recommended).')
segs['hand'] = segs.hand.fillna('R')

trials = segs.merge(videos[['user', 'word', 'stem', 'path']], on=['user', 'word'], how='inner')
trials = trials[trials.word.isin(WORD_IDS)].copy()
trials['cls'] = trials.word.map(WORD2CLS)
trials = trials.sort_values(['stem', 'start']).reset_index(drop=True)

per_video = trials.groupby('stem').size()
print('\ntrials matched to videos:', len(trials), '(paper: 9307)')
print('videos with at least one trial:', len(per_video), '/', len(videos))
print('segments per video (paper says 10-15):', per_video.describe()[['min', '25%', '50%', '75%', 'max']].to_dict())
print('left-handed trials:', int((trials.hand == 'L').sum()), '| right-handed:', int((trials.hand == 'R').sum()),
      '(paper: 1634 / 7673)')
cmp = pd.DataFrame({'ours': trials.groupby('user').size(), 'paper': pd.Series(TABLE6)}).fillna(0).astype(int)
cmp['diff'] = cmp.ours - cmp.paper
print('\ntrials per signer vs Table 6:')
print(cmp.T.to_string())

most common keys in the first JSON files: [('starting', 913), ('ending', 913), ('User', 75), ('Orientation', 75), ('Session', 75), ('View', 75), ('FrameRate', 75), ('FileName', 75), ('no_of_trials', 75), ('trials', 75), ('0', 75), ('1', 75), ('2', 75), ('3', 75), ('4', 75), ('5', 74), ('6', 74), ('7', 73), ('8', 73), ('9', 71), ('RightHand', 62), ('10', 50), ('11', 36), ('12', 26), ('13', 23)]
using keys -> start: starting | end: ending | hand: None
9297 segments parsed
raw hand values: {'RightHand': 7663, 'LeftHand': 1634}
segments with unknown hand: 0

trials matched to videos: 9297 (paper: 9307)
videos with at least one trial: 770 / 774
segments per video (paper says 10-15): {'min': 5.0, '25%': 10.0, '50%': 11.0, '75%': 13.0, 'max': 30.0}
left-handed trials: 1634 | right-handed: 7663 (paper: 1634 / 7673)

trials per signer vs Table 6:
        1    2    3    4    5    6    7    8    9    10   11   12   13   14   15  16   17   18
ours   975  744  583  637  600  815  107  639  656  383

In [7]:
# ---- Cell 7: MediaPipe smoke test, left/right hand check, time estimate ----
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
assert os.path.exists(MODEL_PATH), 'holistic_landmarker.task missing - see the message printed by cell 1.'
print('mediapipe', mp.__version__)
POSE_IDX = [0, 11, 12, 13, 14, 15, 16]   # nose, shoulders, elbows, wrists

def make_landmarker():
    opts = mp_vision.HolisticLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=mp_vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5, min_pose_landmarks_confidence=0.5,
        min_hand_landmarks_confidence=0.5, min_face_detection_confidence=0.5,
        min_face_landmarks_confidence=0.5)
    return mp_vision.HolisticLandmarker.create_from_options(opts)

first = trials.groupby('stem').agg(path=('path', 'first'), f0=('start', 'min'), f1=('end', 'max')).reset_index()
sample = first.sample(min(3, len(first)), random_state=0)

lh_closer_to_pose_left, n_checked, n_frames, t_spent = [], 0, 0, 0.0
for r in sample.itertuples():
    h = make_landmarker()                      # new one per video: timestamps must increase within a landmarker
    cap = cv2.VideoCapture(r.path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    for _ in range(int(r.f0)):
        cap.grab()
    for k in range(60):
        ok, frame = cap.read()
        if not ok:
            break
        rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        t0 = time.time()
        res = h.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), k * 33)
        t_spent += time.time() - t0
        n_frames += 1
        if res.pose_landmarks and res.left_hand_landmarks:
            pl, pr = res.pose_landmarks[15], res.pose_landmarks[16]
            w = res.left_hand_landmarks[0]
            dl = (pl.x - w.x) ** 2 + (pl.y - w.y) ** 2
            dr = (pr.x - w.x) ** 2 + (pr.y - w.y) ** 2
            lh_closer_to_pose_left.append(dl < dr)
    cap.release()
    h.close()
print('video fps (last sample):', fps, '| frame size:', frame.shape if n_frames else None)
fps_proc = n_frames / max(t_spent, 1e-6)
print('MediaPipe speed: %.1f frames/s per process' % fps_proc)

if lh_closer_to_pose_left:
    frac = float(np.mean(lh_closer_to_pose_left))
    SWAP_HANDS = frac < 0.5
    print('left_hand wrist is nearest the pose LEFT wrist in %.0f%% of checked frames -> SWAP_HANDS = %s' % (100 * frac, SWAP_HANDS))
else:
    SWAP_HANDS = False
    print('No frame with both pose and a left hand in the sample; SWAP_HANDS left False. '
          'Only matters for naming - mirroring stays consistent either way.')

procs = max(1, os.cpu_count() or 1)
total_frames = int((first.f1 - first.f0 + 1).sum())
print('frames to process: %d | estimated time with %d processes: %.1f h'
      % (total_frames, procs, total_frames / (fps_proc * procs * 0.8) / 3600))

mediapipe 1.1.1


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792907541.827247      94 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792907541.863991      94 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792907541.868909      95 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792907541.870877      92 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792907541.871256      94 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792907541.894585     

video fps (last sample): 24.08294366956637 | frame size: (720, 1280, 3)
MediaPipe speed: 18.3 frames/s per process
left_hand wrist is nearest the pose LEFT wrist in 100% of checked frames -> SWAP_HANDS = False
frames to process: 673031 | estimated time with 4 processes: 3.2 h


In [8]:
# ---- Cell 8: extract keypoints for every video (resumable; one .npz per video) ----
EXTRACT_SCRIPT = r'''
import sys, json
import numpy as np
from pathlib import Path
from multiprocessing import get_context

POSE_IDX = [0, 11, 12, 13, 14, 15, 16]

def work(job):
    stem, path, f0, f1, swap, out_dir = job
    out = Path(out_dir) / (stem + '.npz')
    if out.exists():
        return stem, -1
    import os
    import cv2
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    cap = cv2.VideoCapture(path)
    for _ in range(f0):
        if not cap.grab():
            break
    coords, masks = [], []
    opts = mp_vision.HolisticLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=os.environ['HOLISTIC_MODEL']),
        running_mode=mp_vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5, min_pose_landmarks_confidence=0.5,
        min_hand_landmarks_confidence=0.5, min_face_detection_confidence=0.5,
        min_face_landmarks_confidence=0.5)
    h = mp_vision.HolisticLandmarker.create_from_options(opts)
    for fi in range(f0, f1 + 1):
        ok, frame = cap.read()
        if not ok:
            break
        H, W = frame.shape[:2]
        if max(H, W) > 640:
            s = 640.0 / max(H, W)
            frame = cv2.resize(frame, (int(W * s), int(H * s)))
        rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        res = h.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), (fi - f0) * 33)
        c = np.zeros((49, 3), np.float32)
        m = np.zeros(49, np.uint8)
        if res.pose_landmarks:
            for k, idx in enumerate(POSE_IDX):
                lm = res.pose_landmarks[idx]
                c[k] = (lm.x, lm.y, lm.z)
                m[k] = 1
        lh, rh = res.left_hand_landmarks, res.right_hand_landmarks
        if swap:
            lh, rh = rh, lh
        for hand, sl, wrist in ((lh, slice(7, 28), 5), (rh, slice(28, 49), 6)):
            if hand:
                arr = np.array([[p.x, p.y, p.z] for p in hand], np.float32)
                if m[wrist]:
                    arr[:, 2] += c[wrist, 2]   # hand z is relative to the wrist; make it comparable to pose z
                c[sl] = arr
                m[sl] = 1
        coords.append(c)
        masks.append(m)
    h.close()
    cap.release()
    coords = np.stack(coords) if coords else np.zeros((0, 49, 3), np.float32)
    masks = np.stack(masks) if masks else np.zeros((0, 49), np.uint8)
    np.savez_compressed(out, coords=coords, mask=masks, f0=f0)
    return stem, len(coords)

if __name__ == '__main__':
    jobs = json.load(open(sys.argv[1]))
    procs = int(sys.argv[2])
    ctx = get_context('spawn')
    done = 0
    with ctx.Pool(procs, maxtasksperchild=20) as pool:
        for stem, n in pool.imap_unordered(work, jobs):
            done += 1
            print('%d/%d %s frames=%d' % (done, len(jobs), stem, n), flush=True)
    print('ALL DONE', flush=True)
'''
(CACHE / 'extract_kp.py').write_text(EXTRACT_SCRIPT)

jobs_df = first.copy()
jobs_df['f0'] = (jobs_df.f0 - 5).clip(lower=0)
jobs_df['f1'] = jobs_df.f1 + 5
jobs_df['stem'] = jobs_df.stem
if QUICK_TEST_VIDEOS:
    jobs_df = jobs_df.head(QUICK_TEST_VIDEOS)
jobs = [[r.stem, r.path, int(r.f0), int(r.f1), bool(SWAP_HANDS), str(KP_DIR)] for r in jobs_df.itertuples()]
json.dump(jobs, open(CACHE / 'jobs.json', 'w'))
todo = [j for j in jobs if not (KP_DIR / (j[0] + '.npz')).exists()]
print(len(jobs), 'videos in total,', len(todo), 'still to extract')

if todo:
    os.environ['HOLISTIC_MODEL'] = MODEL_PATH      # inherited by the extractor's worker processes
    log = open(CACHE / 'extract.log', 'w')
    proc = subprocess.Popen([sys.executable, str(CACHE / 'extract_kp.py'), str(CACHE / 'jobs.json'), str(procs)],
                            stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while proc.poll() is None:          # blocks until finished, so "Save & Run All" works for long runs
        time.sleep(120)
        n_done = len(list(KP_DIR.glob('*.npz')))
        print('%.0f min | %d / %d videos done' % ((time.time() - t0) / 60, n_done, len(jobs)), flush=True)
    print('extractor exit code:', proc.returncode)
    print(''.join(open(CACHE / 'extract.log').readlines()[-5:]))
print('npz files:', len(list(KP_DIR.glob('*.npz'))))

770 videos in total, 770 still to extract
2 min | 0 / 770 videos done
4 min | 4 / 770 videos done
6 min | 7 / 770 videos done
8 min | 9 / 770 videos done
10 min | 12 / 770 videos done
12 min | 15 / 770 videos done
14 min | 19 / 770 videos done
16 min | 21 / 770 videos done
18 min | 25 / 770 videos done
20 min | 28 / 770 videos done
22 min | 32 / 770 videos done
24 min | 35 / 770 videos done
26 min | 38 / 770 videos done
28 min | 43 / 770 videos done
30 min | 46 / 770 videos done
32 min | 50 / 770 videos done
34 min | 53 / 770 videos done
36 min | 59 / 770 videos done
38 min | 63 / 770 videos done
40 min | 68 / 770 videos done
42 min | 71 / 770 videos done
44 min | 74 / 770 videos done
46 min | 80 / 770 videos done
48 min | 84 / 770 videos done
50 min | 88 / 770 videos done
52 min | 93 / 770 videos done
54 min | 97 / 770 videos done
56 min | 99 / 770 videos done
58 min | 102 / 770 videos done
60 min | 106 / 770 videos done
62 min | 110 / 770 videos done
64 min | 114 / 770 videos done
66

In [9]:
# ---- Cell 9: preprocessing functions + self-tests on synthetic data ----
LEFT_BODY, RIGHT_BODY = [1, 3, 5], [2, 4, 6]
PERM = np.arange(N_NODES)
for l, r in zip(LEFT_BODY, RIGHT_BODY):
    PERM[l], PERM[r] = r, l
PERM[7:28], PERM[28:49] = np.arange(28, 49), np.arange(7, 28)

def fill_gaps(coords, mask, max_gap=MAX_GAP):
    # linear interpolation of gaps up to max_gap frames that are bounded by detections on both sides
    coords, mask = coords.copy(), mask.copy()
    for n in range(coords.shape[1]):
        idx = np.flatnonzero(mask[:, n])
        for a, b in zip(idx[:-1], idx[1:]):
            gap = b - a - 1
            if 1 <= gap <= max_gap:
                w = ((np.arange(a + 1, b) - a) / (b - a))[:, None]
                coords[a + 1:b, n] = (1 - w) * coords[a, n] + w * coords[b, n]
                mask[a + 1:b, n] = 1
    return coords, mask

def normalize(coords, mask):
    # centre on mid-shoulder, divide by shoulder width, per frame; missing nodes stay exactly 0
    ls, rs = coords[:, 1], coords[:, 2]
    ok = (mask[:, 1] > 0) & (mask[:, 2] > 0)
    if not ok.any():
        return None, None
    mid = (ls + rs) / 2
    width = np.linalg.norm((ls - rs)[:, :2], axis=1)
    good = ok & (width > 1e-4)
    if not good.any():
        return None, None
    mid_f = np.where(ok[:, None], mid, np.median(mid[ok], axis=0))
    width_f = np.where(good, width, np.median(width[good]))
    out = (coords - mid_f[:, None, :]) / width_f[:, None, None]
    out = out * mask[..., None]
    return out.astype(np.float32), mask

def mirror(coords, mask):
    c = coords[:, PERM].copy()
    c[..., 0] *= -1
    return c, mask[:, PERM]

def resample(coords, mask, t_out=T_OUT):
    # mask-weighted linear interpolation so a detected frame never blends with a zero-filled neighbour
    T = len(coords)
    if T == 1:
        return np.repeat(coords, t_out, 0), np.repeat(mask, t_out, 0)
    src = np.linspace(0, T - 1, t_out)
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, T - 1)
    w = (src - lo)
    wv, wc = w[:, None], w[:, None, None]
    mf = mask.astype(np.float32)
    num = (1 - wc) * (coords[lo] * mf[lo][..., None]) + wc * (coords[hi] * mf[hi][..., None])
    den = (1 - wv) * mf[lo] + wv * mf[hi]
    mask_out = mask[np.rint(src).astype(int)]
    coords_out = np.where(mask_out[..., None] > 0, num / np.maximum(den[..., None], 1e-6), 0.0)
    return coords_out.astype(np.float32), mask_out

# ---- self-tests ----
rng = np.random.default_rng(0)
c = rng.normal(size=(30, N_NODES, 3)).astype(np.float32)
c[:, 1] = [0.6, 0.5, 0.0]; c[:, 2] = [0.4, 0.5, 0.0]          # shoulders: width 0.2
m = np.ones((30, N_NODES), np.uint8)
m[5:8, 10] = 0; m[12:20, 11] = 0; m[0:3, 12] = 0              # short gap, long gap, leading gap
c = c * m[..., None]
c2, m2 = fill_gaps(c, m)
assert m2[5:8, 10].all() and not m2[12:20, 11].any() and not m2[0:3, 12].any()
n, nm = normalize(c2, m2)
assert np.allclose(np.abs(n[:, 1, 0]), 0.5, atol=1e-4)      # shoulders end up at +-0.5
assert np.all(n[nm == 0] == 0)
mc, mm = mirror(n, nm)
mc2, mm2 = mirror(mc, mm)
assert np.allclose(mc2, n) and (mm2 == nm).all()
assert np.allclose(mc[:, 1, 0], -n[:, 2, 0])                 # left/right shoulders swapped and mirrored
const = np.ones((20, N_NODES, 3), np.float32); cm = np.ones((20, N_NODES), np.uint8)
cm[8:14, 20] = 0; const = const * cm[..., None]
rc, rm = resample(const, cm)
assert rc.shape == (T_OUT, N_NODES, 3) and rm.shape == (T_OUT, N_NODES)
assert np.allclose(rc[rm[:, 20] > 0, 20], 1.0)               # no partial values at mask borders
assert np.all(rc[rm == 0] == 0)
print('preprocessing self-tests passed')

preprocessing self-tests passed


In [10]:
# ---- Cell 10: build the cached arrays (two variants) and compare with the paper ----
# 'zero' = detection mask only, short gaps NOT filled (B0 baseline)   'fill' = short gaps interpolated (B1)
def build_variant(trials, fill):
    n = len(trials)
    X = np.zeros((n, 3, T_OUT, N_NODES), np.float32)
    M = np.zeros((n, T_OUT, N_NODES), np.uint8)
    ok = np.zeros(n, bool)
    hand_frames = hand_missing = 0
    cur_stem, kp = None, None
    for i, r in enumerate(trials.itertuples()):
        if r.stem != cur_stem:
            f = KP_DIR / (r.stem + '.npz')
            kp = np.load(f) if f.exists() else None
            if kp is not None:
                kp = dict(coords=kp['coords'], mask=kp['mask'], f0=int(kp['f0']))
            cur_stem = r.stem
        if kp is None:
            continue
        s, e = r.start - kp['f0'], r.end - kp['f0']
        if s < 0 or e >= len(kp['coords']) or e - s < 2:
            continue
        c, m = kp['coords'][s:e + 1], kp['mask'][s:e + 1]
        hand_frames += m[:, 7:].size
        hand_missing += int((m[:, 7:] == 0).sum())
        if fill:
            c, m = fill_gaps(c, m)
        c, m = normalize(c, m)
        if c is None:
            continue
        if r.hand == 'L':
            c, m = mirror(c, m)
        c, m = resample(c, m)
        X[i] = c.transpose(2, 0, 1)
        M[i] = m
        ok[i] = True
    return X, M, ok, hand_missing / max(hand_frames, 1)

t0 = time.time()
Xz, Mz, ok_z, miss_raw = build_variant(trials, fill=False)
Xf, Mf, ok_f, _ = build_variant(trials, fill=True)
keep = ok_z & ok_f
print('built in %.0f s | usable trials: %d / %d' % (time.time() - t0, keep.sum(), len(trials)))
print('hand points missing before filling: %.1f%% (paper: about 41%%)' % (100 * miss_raw))
hand_cols = slice(7, 49)
print('hand points missing after resampling: zero-fill %.1f%% | gap-filled %.1f%%'
      % (100 * (Mz[keep][:, :, hand_cols] == 0).mean(), 100 * (Mf[keep][:, :, hand_cols] == 0).mean()))

meta = trials[keep].reset_index(drop=True)
Xz, Mz, Xf, Mf = Xz[keep], Mz[keep], Xf[keep], Mf[keep]
y = meta.cls.values.astype(np.int64)
np.save(CACHE / 'X_zero.npy', Xz); np.save(CACHE / 'M_zero.npy', Mz)
np.save(CACHE / 'X_fill.npy', Xf); np.save(CACHE / 'M_fill.npy', Mf)
np.save(CACHE / 'y.npy', y)
meta.to_csv(CACHE / 'meta.csv', index=False)

cmp = pd.DataFrame({'ours': meta.groupby('user').size(), 'paper': pd.Series(TABLE6)}).fillna(0).astype(int)
cmp['diff'] = cmp.ours - cmp.paper
print('\ntrials per signer vs Table 6 (diff should be 0 or very small):')
print(cmp.T.to_string())
print('test signers U4+U8 trials:', int(meta.user.isin([4, 8]).sum()), '(paper: 1276)')
print('classes present:', len(set(y)), '/ 60')

built in 25 s | usable trials: 9296 / 9297
hand points missing before filling: 42.2% (paper: about 41%)
hand points missing after resampling: zero-fill 43.1% | gap-filled 39.3%

trials per signer vs Table 6 (diff should be 0 or very small):
        1    2    3    4    5    6    7    8    9    10   11   12   13   14   15  16   17   18
ours   974  744  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  131
paper  975  744  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
diff    -1    0    0    0    0    0    0    0    0    0    0    0    0    0    0   0    0  -10
test signers U4+U8 trials: 1276 (paper: 1276)
classes present: 60 / 60


In [11]:
# ---- Cell 11: split files (P1 official split, 18 leave-one-signer-out folds) ----
users = sorted(int(u) for u in meta.user.unique())
def idx_of(us):
    return np.flatnonzero(meta.user.isin(us).values)

p1 = {'train': idx_of([u for u in users if u not in (4, 5, 8)]), 'val': idx_of([5]), 'test': idx_of([4, 8])}
print('P1 sizes -> train %d | val (U5) %d | test (U4+U8) %d   (paper/plan: ~7431 / 600 / 1276)'
      % (len(p1['train']), len(p1['val']), len(p1['test'])))

loso = {}
for k, s in enumerate(users):
    rest = [u for u in users if u != s]
    v = rest[k % len(rest)]                     # deterministic validation signer for each fold
    loso[s] = {'train': idx_of([u for u in rest if u != v]), 'val': idx_of([v]), 'test': idx_of([s])}

splits = {'P1': {k: v.tolist() for k, v in p1.items()},
          'LOSO': {str(s): {k: v.tolist() for k, v in d.items()} for s, d in loso.items()}}
json.dump(splits, open(CACHE / 'splits.json', 'w'))
print('LOSO folds written:', len(loso))
print('files in cache:', sorted(p.name for p in CACHE.iterdir() if p.is_file()))

P1 sizes -> train 7420 | val (U5) 600 | test (U4+U8) 1276   (paper/plan: ~7431 / 600 / 1276)
LOSO folds written: 18
files in cache: ['M_fill.npy', 'M_zero.npy', 'X_fill.npy', 'X_zero.npy', 'extract.log', 'extract_kp.py', 'jobs.json', 'meta.csv', 'splits.json', 'y.npy']


In [12]:
# ---- Cell 12: 49-node graph, augmentation, bone/motion streams (numpy only) ----
HAND_EDGES = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12),
              (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]
BODY_EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 5), (4, 6)]
EDGES = (BODY_EDGES + [(a + 7, b + 7) for a, b in HAND_EDGES] + [(a + 28, b + 28) for a, b in HAND_EDGES]
         + [(5, 7), (6, 28)])      # hand wrists attached to the body wrists

def build_A(edges, V=N_NODES):
    A = np.eye(V, dtype=np.float32)
    for a, b in edges:
        A[a, b] = A[b, a] = 1
    d = A.sum(1) ** -0.5
    return (d[:, None] * A * d[None, :]).astype(np.float32)

def build_parents(edges, V=N_NODES, root=0):
    adj = collections.defaultdict(list)
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    parent = [-1] * V
    parent[root] = root
    queue = [root]
    while queue:
        u = queue.pop(0)
        for w in adj[u]:
            if parent[w] == -1:
                parent[w] = u
                queue.append(w)
    assert all(p >= 0 for p in parent), 'graph is not connected'
    return np.array(parent)

A_NORM = build_A(EDGES)
PARENT = build_parents(EDGES)

def augment(x, m, rng, p_rot=0.5, p_scale=0.5, p_speed=0.5, p_hdrop=0.2):
    # x: (C, T, V) coordinates, m: (T, V) detection mask. Training only.
    C, T, V = x.shape
    x, m = x.copy(), m.copy()
    if rng.random() < p_speed:                       # speed 0.8-1.2x (nearest-frame time warp)
        s = rng.uniform(0.8, 1.2)
        near = np.rint(np.clip(np.arange(T) * s, 0, T - 1)).astype(int)
        x, m = x[:, near], m[near]
    if rng.random() < p_rot:                         # rotation +-15 degrees in the image plane
        th = np.deg2rad(rng.uniform(-15, 15))
        c, s_ = np.cos(th), np.sin(th)
        x0, x1 = x[0].copy(), x[1].copy()
        x[0], x[1] = c * x0 - s_ * x1, s_ * x0 + c * x1
    if rng.random() < p_scale:                       # scale 0.8-1.2
        x = x * rng.uniform(0.8, 1.2)
    if rng.random() < p_hdrop:                       # one hand disappears for a short span
        sl = slice(7, 28) if rng.random() < 0.5 else slice(28, 49)
        L = int(rng.integers(8, 25))
        st = int(rng.integers(0, max(1, T - L)))
        x[:, st:st + L, sl] = 0
        m[st:st + L, sl] = 0
    return (x * m[None]).astype(np.float32), m

def make_stream(x, m, stream, use_mask, use_z=True):
    # returns features (C', T, V) and the matching mask (T, V)
    if stream == 'joint':
        f, fm = x, m
    elif stream == 'bone':
        f = x - x[:, :, PARENT]
        fm = m * m[:, PARENT]
    elif stream == 'joint_motion':
        f = np.zeros_like(x); fm = np.zeros_like(m)
        f[:, :-1] = x[:, 1:] - x[:, :-1]
        fm[:-1] = m[1:] * m[:-1]
    elif stream == 'bone_motion':
        b, bm = make_stream(x, m, 'bone', False)
        f = np.zeros_like(b); fm = np.zeros_like(bm)
        f[:, :-1] = b[:, 1:] - b[:, :-1]
        fm[:-1] = bm[1:] * bm[:-1]
    else:
        raise ValueError(stream)
    if use_mask:
        f = f * fm[None]
    if not use_z:
        f = f[:2]
    return f.astype(np.float32), fm

# ---- self-tests ----
assert A_NORM.shape == (N_NODES, N_NODES) and np.allclose(A_NORM, A_NORM.T)
assert PARENT[0] == 0 and len(set(PARENT.tolist())) > 10
_rng = np.random.default_rng(1)
_x = _rng.normal(size=(3, T_OUT, N_NODES)).astype(np.float32)
_m = (_rng.random((T_OUT, N_NODES)) > 0.3).astype(np.uint8)
_x = _x * _m[None]
for _s in ('joint', 'bone', 'joint_motion', 'bone_motion'):
    for _um in (False, True):
        _f, _fm = make_stream(_x, _m, _s, _um)
        assert _f.shape == (3, T_OUT, N_NODES) and _fm.shape == (T_OUT, N_NODES)
        if _um:
            assert np.all(_f[:, _fm == 0] == 0)
_a, _am = augment(_x, _m, np.random.default_rng(2), 1, 1, 1, 1)
assert _a.shape == _x.shape and np.all(_a[:, _am == 0] == 0)
print('graph:', len(EDGES), 'edges |', 'self-tests passed')

graph: 51 edges | self-tests passed


In [13]:
# ---- Cell 13: dataset + compact ST-GCN-style model + training loop (PyTorch) ----
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

class BdslDataset(torch.utils.data.Dataset):
    def __init__(self, X, M, y, idx, stream='joint', use_mask=False, aug=False, use_z=True):
        self.X, self.M, self.y, self.idx = X, M, y, idx
        self.stream, self.use_mask, self.aug, self.use_z = stream, use_mask, aug, use_z
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        x, m = np.array(self.X[i]), np.array(self.M[i])
        if self.aug:
            rng = np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,))))
            x, m = augment(x, m, rng)
        f, fm = make_stream(x, m, self.stream, self.use_mask, self.use_z)
        return torch.from_numpy(f), torch.from_numpy(fm.astype(np.float32)), int(self.y[i])

class STBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, dropout=0.2):
        super().__init__()
        self.register_buffer('A', torch.tensor(A, dtype=torch.float32))
        self.B = nn.Parameter(torch.zeros(A.shape, dtype=torch.float32))   # learnable adjacency offset
        self.gcn = nn.Conv2d(cin, cout, 1)
        self.bn1 = nn.BatchNorm2d(cout)
        self.tcn = nn.Sequential(nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)),
                                 nn.BatchNorm2d(cout), nn.Dropout(dropout))
        if cin == cout and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
    def forward(self, x):                                   # x: (N, C, T, V)
        y = self.gcn(x)
        y = torch.einsum('nctv,vw->nctw', y, self.A + self.B)
        y = F.relu(self.bn1(y))
        y = self.tcn(y)
        return F.relu(y + self.res(x))

class GCNNet(nn.Module):
    def __init__(self, in_ch, n_cls, A, V=N_NODES, widths=(64, 64, 128, 128, 256, 256), strides=(1, 1, 2, 1, 2, 1)):
        super().__init__()
        self.data_bn = nn.BatchNorm1d(in_ch * V)
        blocks, c = [], in_ch
        for w, s in zip(widths, strides):
            blocks.append(STBlock(c, w, A, s))
            c = w
        self.blocks = nn.ModuleList(blocks)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(c, n_cls)
    def forward(self, x, mask=None):                        # mask unused here; the missing-aware layer will use it
        N, C, T, V = x.shape
        x = self.data_bn(x.permute(0, 1, 3, 2).reshape(N, C * V, T))
        x = x.reshape(N, C, V, T).permute(0, 1, 3, 2).contiguous()
        for b in self.blocks:
            x = b(x)
        return self.fc(self.drop(x.mean(dim=(2, 3))))

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    logits, ys = [], []
    for x, m, y_ in loader:
        logits.append(model(x.to(device), m.to(device)).float().cpu()); ys.append(y_)
    logits, ys = torch.cat(logits), torch.cat(ys)
    top1 = (logits.argmax(1) == ys).float().mean().item()
    top5 = (logits.topk(5, dim=1).indices == ys[:, None]).any(1).float().mean().item()
    f1 = f1_score(ys.numpy(), logits.argmax(1).numpy(), average='macro')
    return top1, top5, f1

def run_experiment(name, variant, use_mask=False, stream='joint', epochs=40, seed=0, lr=0.1, bs=64,
                   aug=True, use_z=True, split=None):
    split = split or p1
    set_seed(seed)
    X = np.load(CACHE / ('X_%s.npy' % variant), mmap_mode='r')
    M = np.load(CACHE / ('M_%s.npy' % variant), mmap_mode='r')
    tr = BdslDataset(X, M, y, split['train'], stream, use_mask, aug, use_z)
    va = BdslDataset(X, M, y, split['val'], stream, use_mask, False, use_z)
    tl = torch.utils.data.DataLoader(tr, batch_size=bs, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
    vl = torch.utils.data.DataLoader(va, batch_size=128, shuffle=False, num_workers=2)
    model = GCNNet(3 if use_z else 2, 60, A_NORM).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    amp = device == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp)
    best = dict(f1=-1)
    print('%s | stream=%s variant=%s mask=%s | %.2fM params | train %d val %d'
          % (name, stream, variant, use_mask, n_params / 1e6, len(tr), len(va)))
    for ep in range(epochs):
        cur = lr * (ep + 1) / 5 if ep < 5 else 0.5 * lr * (1 + math.cos(math.pi * (ep - 5) / max(1, epochs - 5)))
        for g in opt.param_groups:
            g['lr'] = cur
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, m, y_ in tl:
            x, m, y_ = x.to(device), m.to(device), y_.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp):
                loss = crit(model(x, m), y_)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item() * len(y_); n += len(y_)
        top1, top5, f1 = evaluate(model, vl)
        if f1 > best['f1']:
            best = dict(f1=f1, top1=top1, top5=top5, epoch=ep + 1)
            torch.save(model.state_dict(), CACHE / ('%s_seed%d_best.pt' % (name, seed)))
        if ep % 5 == 0 or ep == epochs - 1:
            print('  ep %3d | loss %.3f | val top1 %.3f top5 %.3f macroF1 %.3f | %.0fs'
                  % (ep + 1, tot / n, top1, top5, f1, time.time() - t0))
    return dict(name=name, seed=seed, params_M=round(n_params / 1e6, 2), best_epoch=best['epoch'],
                val_top1=round(best['top1'], 4), val_top5=round(best['top5'], 4), val_macroF1=round(best['f1'], 4))

# quick shape check on a few real samples before the long runs
_X = np.load(CACHE / 'X_fill.npy', mmap_mode='r'); _M = np.load(CACHE / 'M_fill.npy', mmap_mode='r')
_ds = BdslDataset(_X, _M, y, p1['train'][:8], 'joint', False, True)
_x, _m, _y = next(iter(torch.utils.data.DataLoader(_ds, batch_size=8)))
_net = GCNNet(3, 60, A_NORM)
print('input', tuple(_x.shape), '-> logits', tuple(_net(_x, _m).shape))

device: cuda
input (8, 3, 64, 49) -> logits (8, 60)


In [14]:
# ---- Cell 14: B0 (zero-filled) vs B1 (+ short-gap filling), joint stream, validation signer U5 only ----
EPOCHS = 40          # first pass; raise to 120 for the real runs
results = []
results.append(run_experiment('B0_zerofill_joint', variant='zero', epochs=EPOCHS, seed=0))
results.append(run_experiment('B1_gapfill_joint', variant='fill', epochs=EPOCHS, seed=0))
res_df = pd.DataFrame(results)
res_df.to_csv(CACHE / 'week1_results.csv', index=False)
res_df

B0_zerofill_joint | stream=joint variant=zero mask=False | 1.75M params | train 7420 val 600


/tmp/ipykernel_24/1041029263.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)


  ep   1 | loss 2.860 | val top1 0.573 top5 0.893 macroF1 0.509 | 15s
  ep   6 | loss 1.124 | val top1 0.898 top5 0.980 macroF1 0.879 | 12s
  ep  11 | loss 0.921 | val top1 0.965 top5 0.983 macroF1 0.960 | 13s
  ep  16 | loss 0.850 | val top1 0.953 top5 0.983 macroF1 0.947 | 13s
  ep  21 | loss 0.817 | val top1 0.958 top5 0.983 macroF1 0.954 | 13s
  ep  26 | loss 0.798 | val top1 0.943 top5 0.980 macroF1 0.938 | 13s
  ep  31 | loss 0.783 | val top1 0.957 top5 0.983 macroF1 0.953 | 13s
  ep  36 | loss 0.777 | val top1 0.957 top5 0.983 macroF1 0.953 | 13s
  ep  40 | loss 0.777 | val top1 0.957 top5 0.983 macroF1 0.953 | 13s
B1_gapfill_joint | stream=joint variant=fill mask=False | 1.75M params | train 7420 val 600


/tmp/ipykernel_24/1041029263.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)


  ep   1 | loss 2.874 | val top1 0.562 top5 0.853 macroF1 0.481 | 13s
  ep   6 | loss 1.122 | val top1 0.892 top5 0.983 macroF1 0.872 | 13s
  ep  11 | loss 0.930 | val top1 0.938 top5 0.985 macroF1 0.932 | 13s
  ep  16 | loss 0.858 | val top1 0.953 top5 0.983 macroF1 0.947 | 13s
  ep  21 | loss 0.818 | val top1 0.958 top5 0.983 macroF1 0.954 | 13s
  ep  26 | loss 0.802 | val top1 0.962 top5 0.983 macroF1 0.958 | 13s
  ep  31 | loss 0.784 | val top1 0.960 top5 0.983 macroF1 0.957 | 13s
  ep  36 | loss 0.779 | val top1 0.967 top5 0.983 macroF1 0.964 | 13s
  ep  40 | loss 0.777 | val top1 0.965 top5 0.983 macroF1 0.962 | 13s


,name,seed,params_M,best_epoch,val_top1,val_top5,val_macroF1
0,B0_zerofill_joint,0,1.75,12,0.97,0.9850,0.9671
1,B1_gapfill_joint,0,1.75,33,0.97,0.9833,0.9669


## What to bring back / do next

1. **Gate 1 (plan):** B0 and B1 trained end to end on P1 and validated on U5. The numbers here are single-seed on one signer (about 600 clips), so a 1-2 point gap between B0 and B1 is noise. Run 3 seeds before reading anything into it.
2. **Save the cache:** use *Save Version -> Save & Run All (Commit)*, then create a Kaggle Dataset from the notebook output (`bdslw60_cache`). Later notebooks can load `X_*.npy`, `M_*.npy`, `y.npy`, `meta.csv`, `splits.json` directly and skip the slow extraction.
3. **If a check looked off,** send me the printed output for it: trials vs Table 6 (cell 6/10), hand-missing percentage vs the paper's 41% (cell 10), P1 split sizes (cell 11).
4. **Not in this notebook (still week 1 on your plan):** swap in the official SL-GCN from the SAM-SLR repo, and the literature checks (Hand-Aware Masked GCN, HA-GCN, GCAR).
5. **Never** evaluate U4/U8 here. The test split is only indexed in `splits.json` so it stays untouched until week 3.